In [ ]:
# ============================================================
# ADES QNI-CCP TRAINING — MALIMG DATASET
# Fixed version — all syntax errors corrected:
#   1. compute_gradient_norm: merged lines separated
#   2. quantum_circuit: duplicate @qml.qnode decorator removed
#   3. Image size: 128×128 to match Malimg basic model
#   4. compute_confidence_entropy: uses torch.stack (no backprop)
#   5. compute_mc_dropout_variance: uses torch.stack
#   6. qni_ccp_ades_perturbation: uses torch.stack for S
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import numpy as np
import random
import os
import math
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits    = 6      # Malimg: 6 qubits
num_classes = 25     # Malimg: 25 families
batch_size  = 16
num_epochs  = 70
lr          = 0.0005  # Fixed: was 0.0001 (too slow for convergence)

# ── ADES Hyperparameters ──────────────────────────────────────
EPSILON_MIN = 0.027   # Malimg circuit epsilon: 1/(1+30+6)
LAMBDA_MAX  = 2.0     # Max dynamic range: total range [0.027, 2.027]
MC_T        = 10      # MC Dropout passes

BASIC_CHECKPOINT = "/home/netsec1/Kathan/best_model_final_ch4.pth"
ADES_SAVE_PATH   = "qni_ccp_ades_malimg.pth"

# ========== TRANSFORMS — 128×128 for Malimg ==========
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),   # Augmentation matching basic training
    transforms.RandomRotation(10),
    transforms.Resize((128, 128)),       # Malimg resolution
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((128, 128)),       # Malimg resolution
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASETS ==========
TRAIN_PATH = '/home/netsec1/Kathan/dataset_folder/malimg_dataset/train'
VAL_PATH   = '/home/netsec1/Kathan/dataset_folder/malimg_dataset/val'
TEST_PATH  = '/home/netsec1/Kathan/dataset_folder/malimg_dataset/test'

train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ========== CLASS WEIGHTS ==========
labels_list          = [label for _, label in train_dataset.samples]
class_weights        = compute_class_weight('balanced',
                                             classes=np.unique(labels_list),
                                             y=labels_list)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights computed.")

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                          shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
# FIX: Only ONE @qml.qnode decorator — your code had two stacked.
# Using interface="torch" WITHOUT backprop — matches your Malimg
# basic model and allows torch.stack per-sample forward pass.
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")   # Single decorator — fixed
def quantum_circuit(inputs, weights):
    # Per-sample inputs[i] — used with torch.stack forward
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)          # Scalar per qubit
    for l in range(weights.shape[0]):        # 3 variational layers
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)  # Trainable rotation
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])        # Linear entanglement
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (3, n_qubits)}  # 3 layers × 6 qubits

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    """
    CNN: 128×128 → 6-dim latent vector.
    Single fc layer (128→6) — matches best_model_final_ch4.pth.
    Dropout=0.4 in CNN encoder — same as basic model training.
    Dropout layers stay active in model.train() → enables MC Dropout.
    """
    def __init__(self, final_dim, dropout=0.4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),     # 128→64
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(8, 16, 3, stride=2, padding=1),    # 64→32
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(16, 32, 3, stride=2, padding=1),   # 32→16
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),   # 16→8
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # 8→4
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))                  # → [batch,128,1,1]
        )
        self.fc = nn.Linear(128, final_dim)               # 128→6

    def forward(self, x):
        x = self.conv(x)              # [batch, 128, 1, 1]
        x = x.view(x.size(0), -1)    # [batch, 128]
        return self.fc(x)             # [batch, 6]


class HybridQNN(nn.Module):
    """
    Hybrid model: CNN → tanh → QNN (torch.stack) → Classifier.
    Matches best_model_final_ch4.pth architecture exactly.
    Dropout in CNN (0.4) and classifier (0.3) enables MC Dropout
    uncertainty estimation via model.train() stochastic passes.
    """
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer           = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier        = nn.Sequential(
            nn.Linear(n_qubits, 64), nn.ReLU(), nn.Dropout(0.3),  # 6→64
            nn.Linear(64, 32),       nn.ReLU(), nn.Dropout(0.3),  # 64→32
            nn.Linear(32, 16),       nn.ReLU(),                   # 32→16
            nn.Linear(16, num_classes)                             # 16→25
        )

    def forward(self, x):
        x     = torch.tanh(self.feature_extractor(x))          # [batch, 6]
        q_out = torch.stack([self.q_layer(f) for f in x])      # [batch, 6]
        return self.classifier(q_out)                          # [batch, 25]


# ========== FOCAL LOSS ==========
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=0.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


# ========== ADES CORE FUNCTIONS ==========

def compute_gradient_norm(model, features, labels):
    """
    Cue 1: Gradient norm via classifier-only path.
    FIX: Separated the merged return+assignment line.
    Uses torch.stack (no backprop) — consistent with Malimg circuit.
    interface="torch" blocks quantum gradient propagation,
    so we skip the quantum layer and go classifier-only for S.
    High grad norm → near decision boundary → higher epsilon.
    Returns: grad_norms [batch] — L2 norm per sample.
    """
    # FIX: This is now correctly a separate assignment (was merged with return)
    feats_grad = features.clone().detach().requires_grad_(True)  # [batch, 6]

    # Classifier-only path — fully differentiable, no quantum blockage
    logits = model.classifier(feats_grad)       # [batch, 25] skip quantum
    loss   = F.cross_entropy(logits, labels)    # Scalar
    loss.backward()                             # ∂loss/∂feats_grad populated

    if feats_grad.grad is None:
        return torch.zeros(features.size(0), device=features.device)

    # L2 norm per sample: [batch, 6] → [batch]
    grad_norms = torch.norm(feats_grad.grad.data, p=2, dim=1)
    return grad_norms   # FIX: This return is now on its own line


def compute_confidence_entropy(model, features):
    """
    Cue 2: Shannon entropy of softmax output.
    Uses torch.stack — consistent with Malimg forward pass.
    High entropy → uncertain → lower epsilon (don't overwhelm).
    Low entropy → confident → higher epsilon (needs harder push).
    Returns: entropy [batch] — higher means more uncertain.
    """
    with torch.no_grad():
        # torch.stack per-sample — matches HybridQNN.forward()
        q_out  = torch.stack([model.q_layer(f) for f in features])  # [batch, 6]
        logits = model.classifier(q_out)                             # [batch, 25]
        probs  = F.softmax(logits, dim=1)                            # [batch, 25]

        # H = -sum(p * log(p)) — eps prevents log(0)
        entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)      # [batch]
    return entropy


def compute_mc_dropout_variance(model, features, T=10):
    """
    Cue 3: MC Dropout variance across T stochastic forward passes.
    Uses torch.stack — consistent with Malimg forward pass.
    model.train() keeps dropout active → stochastic predictions.
    High variance → uncertain → lower epsilon.
    Low variance → certain → higher epsilon (needs harder push).
    Returns: mc_variance [batch] — mean variance across classes.
    """
    all_probs = []

    model.train()   # Activates dropout — CRITICAL for stochastic passes
    with torch.no_grad():
        for t in range(T):
            # torch.stack per-sample — matches HybridQNN.forward()
            q_out  = torch.stack([model.q_layer(f) for f in features])  # [batch, 6]
            logits = model.classifier(q_out)                             # [batch, 25]
            probs  = F.softmax(logits, dim=1)                            # [batch, 25]
            all_probs.append(probs.unsqueeze(0))                         # [1, batch, 25]

    all_probs_stacked = torch.cat(all_probs, dim=0)          # [T, batch, 25]
    mc_variance = all_probs_stacked.var(dim=0).mean(dim=1)   # [batch]

    model.eval()    # Restore eval mode after MC passes
    return mc_variance


def normalize_to_01(tensor):
    """Min-max normalize to [0,1]. Returns 0.5 for degenerate batches."""
    t_min = tensor.min()
    t_max = tensor.max()
    if (t_max - t_min) < 1e-8:
        return torch.full_like(tensor, 0.5)
    return (tensor - t_min) / (t_max - t_min)


def compute_ades_epsilon(model, x, labels, epsilon_min, lambda_max, T=10):
    """
    Compute per-sample adaptive epsilon.
    Formula: epsilon_x = epsilon_min + lambda_max * sigma(x)
    sigma(x) = (norm_grad + (1-norm_entropy) + (1-norm_variance)) / 3
    Returns: epsilon_per_sample [batch,1], sigma_stats dict.
    """
    model.eval()

    with torch.no_grad():
        features = model.feature_extractor(x)   # [batch, 6]
        features = torch.tanh(features)          # [batch, 6]

    # Cue 1: gradient norm (classifier-only, no quantum blockage)
    features_for_grad = features.clone().detach().requires_grad_(True)
    grad_norms        = compute_gradient_norm(model, features_for_grad, labels)

    # Cue 2: confidence entropy
    entropy    = compute_confidence_entropy(model, features)

    # Cue 3: MC Dropout variance
    mc_variance = compute_mc_dropout_variance(model, features, T=T)

    # Normalize all three to [0,1]
    norm_grad     = normalize_to_01(grad_norms)
    norm_entropy  = normalize_to_01(entropy)
    norm_variance = normalize_to_01(mc_variance)

    # sigma: high grad + low entropy + low variance = higher epsilon
    sigma = (norm_grad + (1.0 - norm_entropy) + (1.0 - norm_variance)) / 3.0

    epsilon_per_sample = epsilon_min + lambda_max * sigma   # [batch]
    epsilon_per_sample = epsilon_per_sample.unsqueeze(1)    # [batch, 1]

    sigma_stats = {
        "eps_mean"  : epsilon_per_sample.mean().item(),
        "eps_min"   : epsilon_per_sample.min().item(),
        "eps_max"   : epsilon_per_sample.max().item(),
        "eps_std"   : epsilon_per_sample.std().item(),
        "sigma_mean": sigma.mean().item(),
    }
    return epsilon_per_sample.detach(), sigma_stats


def qni_ccp_ades_perturbation(model, x, y, centroids,
                               epsilon_min, lambda_max, T=10):
    """
    ADES QNI-CCP perturbation.
    Formula: z' = z + epsilon_x * [S ⊙ (μ_c' − z)]
    Uses classifier-only sensitivity S (quantum blocks gradients).
    Uses torch.stack for QNI pass (matches Malimg circuit).
    Returns: perturbed_features [batch,6] detached, sigma_stats dict.
    """
    model.eval()

    # Step 1: Clean features
    with torch.no_grad():
        original_features = torch.tanh(model.feature_extractor(x))  # [batch, 6]

    # Step 2: ADES per-sample epsilon
    epsilon_per_sample, sigma_stats = compute_ades_epsilon(
        model, x, y, epsilon_min, lambda_max, T=T
    )

    # Step 3: Sensitivity S — classifier-only (quantum blocks grad)
    features_for_grad = original_features.clone().detach().requires_grad_(True)
    logits = model.classifier(features_for_grad)    # Skip quantum layer
    loss   = F.cross_entropy(logits, y)
    loss.backward()
    S = features_for_grad.grad.data if features_for_grad.grad is not None \
        else torch.zeros_like(features_for_grad)    # [batch, 6]

    # Step 4: Random wrong-class target
    target_classes = []
    for i in range(y.size(0)):
        available = [c for c in range(centroids.size(0)) if c != y[i].item()]
        target_classes.append(
            random.choice(available) if available
            else (y[i].item() + 1) % centroids.size(0)
        )
    target_tensor = torch.tensor(target_classes, device=y.device)
    mu_c_prime    = centroids[target_tensor]                    # [batch, 6]

    # Step 5: Apply ADES-scaled perturbation
    direction          = mu_c_prime - original_features         # [batch, 6]
    weighted_direction = S * direction                          # [batch, 6]
    perturbed          = original_features + \
                         epsilon_per_sample * weighted_direction  # [batch, 6]

    return perturbed.detach(), sigma_stats


def compute_class_centroids(model, loader, device, num_classes):
    """Compute μ_c = mean tanh(CNN(x)) per class. Shape: [25, 6]."""
    model.eval()
    sums   = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in loader:
            x, y     = x.to(device), y.to(device)
            features = torch.tanh(model.feature_extractor(x))
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c]   += features[mask].sum(0)
                    counts[c] += mask.sum()

    counts[counts == 0] = 1
    return sums / counts.unsqueeze(1)   # [25, 6]


def evaluate(model, dataloader, loss_fn, device):
    """Standard evaluation — val loss and accuracy."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            loss           = loss_fn(outputs, labels)
            total_loss    += loss.item()
            _, predicted   = torch.max(outputs.data, 1)
            total         += labels.size(0)
            correct       += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# ========== MAIN EXECUTION ==========

print("\nInitializing ADES QNI-CCP Training — Malimg...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)

if os.path.exists(BASIC_CHECKPOINT):
    ckpt  = torch.load(BASIC_CHECKPOINT, map_location=device)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and \
            'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    print(f"  Loaded: {BASIC_CHECKPOINT}")
else:
    print(f"  ⚠️  Checkpoint not found — training from scratch.")

print("Computing class centroids...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)
loss_fn   = FocalLoss(alpha=1, gamma=0.0)   # Start Stage 1: CE
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

best_val_acc               = 0.0
epochs_without_improvement = 0
early_stopping_patience    = 7

print(f"\nStarting ADES Training | {num_epochs} epochs")
print(f"  epsilon_min={EPSILON_MIN} | lambda_max={LAMBDA_MAX} | MC_T={MC_T}")
print(f"  Range: [{EPSILON_MIN:.4f}, {EPSILON_MIN+LAMBDA_MAX:.4f}]")
print("="*60)

for epoch in range(num_epochs):

    # Two-stage focal loss
    if epoch < 20:
        loss_mode     = "Stage 1: CE (Gamma=0) + ADES"
        loss_fn.gamma = 0.0
    else:
        loss_mode     = "Stage 2: Focal (Gamma=1) + ADES"
        loss_fn.gamma = 1.0

    # Recompute centroids every 5 epochs
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  Centroids updated at epoch {epoch}")

    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    epoch_eps_mean, epoch_sigma_mean = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Clean pass
        optimizer.zero_grad()
        logits_clean = model(inputs)
        loss_clean   = loss_fn(logits_clean, labels)

        # 2. ADES perturbation
        perturbed_features, sigma_stats = qni_ccp_ades_perturbation(
            model, inputs, labels, centroids,
            epsilon_min = EPSILON_MIN,
            lambda_max  = LAMBDA_MAX,
            T           = MC_T
        )

        # 3. QNI pass — torch.stack matches Malimg circuit
        model.train()
        q_out_p    = torch.stack([model.q_layer(f) for f in perturbed_features])
        logits_qni = model.classifier(q_out_p)
        loss_qni   = loss_fn(logits_qni, labels)

        # 4. Centroid regularization
        clean_feats       = torch.tanh(model.feature_extractor(inputs))
        current_centroids = centroids[labels]
        loss_centroid     = F.mse_loss(clean_feats, current_centroids)

        # 5. Combined loss
        loss = 0.6 * loss_clean + 0.2 * loss_qni + 0.2 * loss_centroid

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss    += loss.item()
        _, predicted     = torch.max(logits_clean, 1)
        running_correct += (predicted == labels).sum().item()
        total_samples   += labels.size(0)
        epoch_eps_mean.append(sigma_stats["eps_mean"])
        epoch_sigma_mean.append(sigma_stats["sigma_mean"])
        loop.set_postfix(loss=loss.item(), eps=f"{sigma_stats['eps_mean']:.3f}")

    train_loss = running_loss    / len(train_loader)
    train_acc  = running_correct / total_samples
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    scheduler.step(val_acc)

    avg_eps   = np.mean(epoch_eps_mean)
    avg_sigma = np.mean(epoch_sigma_mean)

    print(f"Epoch {epoch+1}/{num_epochs} | {loss_mode}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    print(f"   ADES | avg_epsilon={avg_eps:.4f} | avg_sigma={avg_sigma:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), ADES_SAVE_PATH)
        print(f"   Best model saved → {ADES_SAVE_PATH}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= early_stopping_patience:
            print(f"   Early stopping at epoch {epoch+1}")
            break

print(f"\nADES Training complete. Best val_acc: {best_val_acc:.4f}")
print(f"Model saved: {ADES_SAVE_PATH}")

Using device: cuda
Train: 7459 | Val: 923 | Test: 961
Class weights computed.

Initializing ADES QNI-CCP Training — Malimg...
  Loaded: /home/netsec1/Kathan/best_model_final_ch4.pth
Computing class centroids...

Starting ADES Training | 70 epochs
  epsilon_min=0.027 | lambda_max=2.0 | MC_T=10
  Range: [0.0270, 2.0270]


Epoch 1/70:   0%|          | 0/467 [00:00<?, ?it/s]

Epoch 1/70 | Stage 1: CE (Gamma=0) + ADES
   Train Loss: 1.0662 | Train Acc: 0.7640 | Val Acc: 0.8418
   ADES | avg_epsilon=1.0457 | avg_sigma=0.5093
   Best model saved → qni_ccp_ades_malimg.pth


Epoch 2/70:   0%|          | 0/467 [00:00<?, ?it/s]

Epoch 2/70 | Stage 1: CE (Gamma=0) + ADES
   Train Loss: 0.5308 | Train Acc: 0.7913 | Val Acc: 0.9339
   ADES | avg_epsilon=1.0329 | avg_sigma=0.5030
   Best model saved → qni_ccp_ades_malimg.pth


...

Epoch 23/70:   0%|          | 0/467 [00:00<?, ?it/s]

Epoch 23/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.1703 | Train Acc: 0.9224 | Val Acc: 0.9686
   ADES | avg_epsilon=1.1354 | avg_sigma=0.5542
   Best model saved → qni_ccp_ades_malimg.pth


Epoch 24/70:   0%|          | 0/467 [00:00<?, ?it/s]

Epoch 24/70 | Stage 2: Focal (Gamma=1) + ADES
   Train Loss: 0.1674 | Train Acc: 0.9221 | Val Acc: 0.9675
   ADES | avg_epsilon=1.1381 | avg_sigma=0.5556


Epoch 25/70:   0%|          | 0/467 [00:00<?, ?it/s]